# MO433 on Colab

Thin runner: all the code lives in the repository and every step is a `make` target.
Run the cells from top to bottom; only the **last cell** changes from session to session.

Before the first run:
*Runtime → Change runtime type → GPU: **L4***. All the timed runs of the report (throughput,
teacher cache, trainings) used the L4 (report Sec. 5.3); another GPU gives the same kind of
results, but its timings and GPU-hours are not comparable, and `train.py` prints a warning.

On Colab, `make` only builds what is **missing** on Drive: after cloning, the code files get
an old fixed date, so they never look newer than the data. To redo a step, delete its output
file. Never use `make -B` here: it rebuilds the whole chain of prerequisites.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
# The data folder on Drive: the original results, or "mo433-2s2026/repro" for the
# from-scratch reproduction (README, "From-scratch check"; its laptop part is uploaded there first)
DATA = "mo433-2s2026"
os.environ["MO433_DATA"] = f"/content/drive/MyDrive/{DATA}"
assert os.path.isdir(os.environ["MO433_DATA"]), f"Drive folder {DATA} not found"

In [ ]:
# Clone (or update) the public repository.
import subprocess

REPO = "https://github.com/helen-fornazier/mo433-2s2026.git"
CODE = "/content/mo433-2s2026"

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True,
                          text=True).stdout.strip()

if not os.path.isdir(CODE):
    git("clone", "-q", REPO, CODE)
else:
    git("pull", "-q", "origin", "main", cwd=CODE)
print("commit:", git("log", "--oneline", "-1", cwd=CODE))

# Old fixed date on every code file: make then only builds outputs that are missing on Drive.
!find {CODE} -path {CODE}/.git -prune -o -exec touch -h -d 2000-01-01 {{}} +
%cd {CODE}

In [ ]:
# Colab's own torch/torchvision (CUDA build) are kept; the pinned libraries are installed.
!pip install -q -r requirements.txt
!pip install -q -e .
!make env-info

## Run a step

Set `TARGET` to the step to run (see `make help` and the README). `make -n` first shows what
would run; check it before spending GPU time.

In [ ]:
TARGET = "env-info"
!make -n {TARGET}

In [ ]:
!make {TARGET}